In [16]:
import os
import cv2
import joblib
import numpy as np

from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report


# =========================
# 1) إعداد المسارات
# =========================

dataset_path = "dataset"

classes = ["class1", "class2", "class3"]

X = []
y = []



In [18]:

# =========================
# 2) دالة استخراج HOG
# =========================

def extract_hog_features(image_path):
    img = cv2.imread(image_path)

    # تحويل إلى رمادي
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # توحيد حجم الصور
    gray = cv2.resize(gray, (128, 128))

    # استخراج HOG
    features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys"
    )

    return features



In [20]:

# =========================
# 3) قراءة الصور من المجلدات
# =========================

for class_name in classes:
    class_folder = os.path.join(dataset_path, class_name)

    for image_name in os.listdir(class_folder):
        image_path = os.path.join(class_folder, image_name)

        try:
            features = extract_hog_features(image_path)
            X.append(features)
            y.append(class_name)
        except:
            print("Error with image:", image_path)


X = np.array(X)
y = np.array(y)

print("Number of images:", len(X))
print("Feature vector size:", X.shape)



Number of images: 69
Feature vector size: (69, 8100)


In [21]:

# =========================
# 4) تقسيم البيانات
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [22]:


# =========================
# 5) تدريب نموذج التصنيف
# =========================

model = SVC(kernel="linear")

model.fit(X_train, y_train)


SVC(kernel='linear')

In [26]:


# =========================
# 6) التنبؤ
# =========================

y_pred = model.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))



Confusion Matrix:
[[4 0 0]
 [2 2 1]
 [0 3 2]]

Classification Report:
              precision    recall  f1-score   support

      class1       0.67      1.00      0.80         4
      class2       0.40      0.40      0.40         5
      class3       0.67      0.40      0.50         5

    accuracy                           0.57        14
   macro avg       0.58      0.60      0.57        14
weighted avg       0.57      0.57      0.55        14



In [28]:

# =========================
# 7) حفظ النموذج
# =========================

joblib.dump(model, "hog_svm_model.pkl")

print("Model saved successfully.")



Model saved successfully.


In [30]:

# =========================
# 8) إعادة تحميل النموذج
# =========================

loaded_model = joblib.load("hog_svm_model.pkl")

y_pred_loaded = loaded_model.predict(X_test)

print("\nConfusion Matrix After Loading Model:")
print(confusion_matrix(y_test, y_pred_loaded))




Confusion Matrix After Loading Model:
[[4 0 0]
 [2 2 1]
 [0 3 2]]


In [45]:

# =========================
# 9) تجربة صورة جديدة
# =========================

new_image_path = "test3.png"

new_features = extract_hog_features(new_image_path)
new_features = new_features.reshape(1, -1)

prediction = loaded_model.predict(new_features)

print("Predicted Class:", prediction[0])

Predicted Class: class1
